Partimos de oito chunks técnicos sobre APIs e webhooks. Vamos consultar a mesma pergunta com BM25 e busca vetorial e combinar as posições com RRF. Não há reranking nesta prática.

As listas vêm da execução dos modelos e não foram ajustadas para repetir as visualizações do artigo. O Qdrant roda em memória, sem serviço externo. A execução exibida usa qdrant-client 1.15.1, fastembed 0.8.1 e sentence-transformers 5.1.0. Para usar outro kernel, instale essas dependências nele com `%pip install qdrant-client==1.15.1 fastembed==0.8.1 sentence-transformers==5.1.0` antes de executar as células.


## 1. Preparando os chunks

Mantemos os identificadores já conhecidos no artigo. Os trechos cobrem o erro exato, o resultado desconhecido de uma entrega, idempotência e tópicos próximos que também podem aparecer entre os candidatos.


In [10]:
corpus = [
    (42, "ECONNRESET na entrega",
     "ECONNRESET indica que a conexão foi encerrada antes de uma resposta HTTP completa. "
     "O emissor não recebeu confirmação do webhook."),
    (43, "Resultado desconhecido",
     "Quando a resposta se perde, o emissor não sabe se o consumidor recebeu ou processou o evento. "
     "A nova tentativa deve considerar que o primeiro envio pode ter produzido efeitos."),
    (44, "Backoff entre tentativas",
     "Backoff exponencial aumenta o intervalo entre retentativas de webhooks "
     "e reduz a pressão sobre um serviço temporariamente instável."),
    (77, "Idempotência e duplicidade",
     "O consumidor registra o event_id antes de executar efeitos. "
     "Se o mesmo evento chegar novamente, ele reconhece a entrega anterior "
     "e evita cobrança ou notificação duplicada."),
    (118, "Falha de socket em HTTP",
     "Uma falha de socket pode interromper uma chamada HTTP em andamento "
     "sem revelar ao cliente o resultado da operação. Registre a tentativa "
     "e investigue o estado antes de repetir a chamada."),
    (205, "Rastreando entregas",
     "Logs de webhooks registram event_id, tentativa, resposta e motivo da próxima ação. "
     "O histórico ajuda a investigar entregas sem confirmação."),
    (301, "Rate limit e capacidade",
     "Uma resposta 429 indica que o consumidor recebeu mais eventos do que consegue processar. "
     "O provedor deve reduzir o ritmo e respeitar a política de espera."),
    (302, "Autenticação da API",
     "A autenticação de uma API valida credenciais e permissões antes de aceitar chamadas. "
     "Tokens expirados precisam ser renovados antes de repetir uma requisição."),
]
chunks = [{"id": id, "title": title, "text": text} for id, title, text in corpus]

print(f"chunks preparados: {len(chunks)}")
for chunk in chunks:
    print(f"Chunk {chunk['id']:03d} | {chunk['title']}")


chunks preparados: 8
Chunk 042 | ECONNRESET na entrega
Chunk 043 | Resultado desconhecido
Chunk 044 | Backoff entre tentativas
Chunk 077 | Idempotência e duplicidade
Chunk 118 | Falha de socket em HTTP
Chunk 205 | Rastreando entregas
Chunk 301 | Rate limit e capacidade
Chunk 302 | Autenticação da API


## 2. Indexando os dois sinais

O modelo denso representa o significado do trecho. O BM25 gera vetores esparsos baseados nos termos e o Qdrant aplica IDF na coleção. Cada ponto guarda as duas representações do mesmo chunk. O modelo denso é multilíngue porque a pergunta e os textos estão em português.


In [11]:
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

dense_model = SentenceTransformer("intfloat/multilingual-e5-small")
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25", language="portuguese")
texts = [chunk["text"] for chunk in chunks]
dense_vectors = dense_model.encode([f"passage: {text}" for text in texts])
sparse_vectors = list(sparse_model.passage_embed(texts))

client = QdrantClient(":memory:")
collection = "webhooks-hybrid-rrf"
client.create_collection(
    collection_name=collection,
    vectors_config={"dense": models.VectorParams(size=dense_vectors.shape[1], distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
)
client.upsert(
    collection_name=collection,
    points=[
        models.PointStruct(
            id=chunk["id"],
            vector={
                "dense": dense.tolist(),
                "sparse": models.SparseVector(indices=sparse.indices.tolist(), values=sparse.values.tolist()),
            },
            payload={"title": chunk["title"], "text": chunk["text"]},
        )
        for chunk, dense, sparse in zip(chunks, dense_vectors, sparse_vectors)
    ],
)

print(f"pontos indexados: {client.count(collection_name=collection, exact=True).count}")
print(f"dimensões do vetor denso: {dense_vectors.shape[1]}")
print("representações por ponto: dense + sparse BM25")


pontos indexados: 8
dimensões do vetor denso: 384
representações por ponto: dense + sparse BM25


## 3. Consultando cada índice separadamente

A mesma pergunta passa pelos dois modelos. Cada busca devolve seu próprio top-4 e seu próprio score. Vamos preservar essas listas para enxergar o que a fusão receberá.


In [12]:
query = "Recebi ECONNRESET ao enviar um webhook. Posso tentar novamente sem duplicar o evento?"
query_dense = dense_model.encode([f"query: {query}"])[0].tolist()
query_sparse_embedding = list(sparse_model.query_embed([query]))[0]
query_sparse = models.SparseVector(
    indices=query_sparse_embedding.indices.tolist(),
    values=query_sparse_embedding.values.tolist(),
)
top_k = 4

dense_hits = client.query_points(collection, query=query_dense, using="dense", limit=top_k).points
sparse_hits = client.query_points(collection, query=query_sparse, using="sparse", limit=top_k).points

for label, hits in (("BUSCA LEXICAL (BM25)", sparse_hits), ("BUSCA VETORIAL", dense_hits)):
    print(label)
    for position, hit in enumerate(hits, start=1):
        print(f"#{position} Chunk {hit.id:03d} | score={hit.score:.4f} | {hit.payload['title']}")
    print()


BUSCA LEXICAL (BM25)
#1 Chunk 043 | score=8.0068 | Resultado desconhecido
#2 Chunk 042 | score=7.4219 | ECONNRESET na entrega
#3 Chunk 077 | score=6.5163 | Idempotência e duplicidade
#4 Chunk 301 | score=3.0720 | Rate limit e capacidade

BUSCA VETORIAL
#1 Chunk 042 | score=0.8991 | ECONNRESET na entrega
#2 Chunk 205 | score=0.8732 | Rastreando entregas
#3 Chunk 043 | score=0.8652 | Resultado desconhecido
#4 Chunk 077 | score=0.8648 | Idempotência e duplicidade



## 4. Combinando posições com RRF

Com as duas listas recuperadas pelo Qdrant, calculamos o RRF diretamente pelas posições. Cada aparição do chunk contribui com `1 / (60 + posição)`, usando posições a partir de #1 como no artigo. Somamos essas contribuições, sem misturar os scores lexical e vetorial.


In [13]:
sparse_positions = {hit.id: position for position, hit in enumerate(sparse_hits, start=1)}
dense_positions = {hit.id: position for position, hit in enumerate(dense_hits, start=1)}
rrf_k = 60
scores = {}
for positions in (sparse_positions, dense_positions):
    for chunk_id, position in positions.items():
        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (rrf_k + position)

fused_ids = sorted(scores, key=lambda chunk_id: (-scores[chunk_id], chunk_id))
print("LISTA COMBINADA — RRF (k = 60)")
for position, chunk_id in enumerate(fused_ids, start=1):
    origins = []
    if chunk_id in sparse_positions:
        origins.append(f"lexical #{sparse_positions[chunk_id]}")
    if chunk_id in dense_positions:
        origins.append(f"vetorial #{dense_positions[chunk_id]}")
    print(f"#{position} Chunk {chunk_id:03d} | {' + '.join(origins)} | RRF={scores[chunk_id]:.5f}")


LISTA COMBINADA — RRF (k = 60)
#1 Chunk 042 | lexical #2 + vetorial #1 | RRF=0.03252
#2 Chunk 043 | lexical #1 + vetorial #3 | RRF=0.03227
#3 Chunk 077 | lexical #3 + vetorial #4 | RRF=0.03150
#4 Chunk 205 | vetorial #2 | RRF=0.01613
#5 Chunk 301 | lexical #4 | RRF=0.01562


## 5. Lendo o resultado

Agora há uma lista única de candidatos. Um chunk encontrado pelos dois caminhos recebe duas contribuições; um chunk presente em apenas uma lista recebe uma. A ordem concreta depende dos textos e dos modelos usados nesta execução, não dos scores ilustrativos do artigo.

A prática mostrou o mecanismo de recuperação e fusão, mas não demonstrou que a lista combinada responde melhor à pergunta. Também não reavaliamos o conteúdo de cada candidato: esse será o papel do ColBERT na segunda prática. A comparação de qualidade ficará para o artigo 7.
